# N-Back Working Memory

**Track:** Executive Functions
**Construct:** Working memory updating

Tests the ability to continuously update and monitor working memory contents, matching current items against items N positions back in a sequence.

## Cognitive Science Background

The **n-back task** (Kirchner, 1958) is a standard measure of working memory updating — one of the three core executive functions (Miyake et al., 2000). As N increases, demands on maintenance, updating, and interference resolution grow. Transformation variants add an additional processing step, requiring manipulation of held items before comparison.

**Human baseline:** ~90% at 2-back, ~70% at 3-back, ~50% at 4-back (Kane et al., 2007).

## Methodology

Six conditions of increasing difficulty test working memory capacity and manipulation:

- **2-back:** Baseline working memory monitoring
- **3-back:** Standard difficulty
- **4-back:** High capacity demand
- **4-back transform:** Match after applying a transformation rule (next consonant)
- **5-back:** Near-ceiling capacity demand
- **5-back transform:** Maximum difficulty — far maintenance plus transformation

Items are presented in batch format without revealing the n-back letter. Lure trials (~12%) match at N±1 but not at N, testing interference resistance.

## Scoring

Signal detection (d-prime) scoring weighted toward harder conditions. Higher scores indicate greater working memory capacity and updating efficiency.

### References

Kirchner (1958), Miyake et al. (2000), Kane et al. (2007), Owen et al. (2005)

In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


In [ ]:
# --- Inlined stimuli ---
"""
N-back Working Memory v3 — Harder variant for LLMs.

Changes from v2:
- Levels: 2-back, 3-back, 4-back, 5-back (dropped trivial 1-back)
- Batch presentation: model sees a segment of the sequence and must answer for ALL positions
  (no more giving away the N-back letter in the prompt)
- Transformation N-back: at 4-back and 5-back, half the segments require checking if the
  current letter is the NEXT letter in the alphabet from the N-back letter
- Lure trials: ~15% of non-targets match at N±1 (not N), testing precision of position tracking
- Longer sequences: 80 items for 4-back and 5-back
"""

import random
import hashlib


CONSONANTS = list("BCDFGHJKLMNPQRSTVWXZ")

# For transformation n-back: next letter mapping (wraps Z->B since we skip vowels)
_NEXT_CONSONANT = {}
for i, c in enumerate(CONSONANTS):
    _NEXT_CONSONANT[c] = CONSONANTS[(i + 1) % len(CONSONANTS)]


def generate_nback_v3(n_level: int, length: int, seed: str,
                       target_rate: float = 0.22, lure_rate: float = 0.12,
                       transform: bool = False):
    """
    Generate an N-back sequence with optional lure trials and transformation rule.
    
    Args:
        n_level: N value (2-5)
        length: total items
        seed: string seed for reproducibility
        target_rate: fraction of targetable positions that are targets
        lure_rate: fraction of non-targets that are lures (match at N±1)
        transform: if True, target = "next consonant after N-back letter"
    """
    rng = random.Random(int(hashlib.sha256(seed.encode()).hexdigest(), 16))
    
    n_targetable = length - n_level
    n_targets = int(n_targetable * target_rate)
    n_lures = int((n_targetable - n_targets) * lure_rate)
    
    # Assign positions
    targetable = list(range(n_level, length))
    rng.shuffle(targetable)
    target_positions = set(targetable[:n_targets])
    lure_candidates = [p for p in targetable[n_targets:] if p > n_level]  # lures need N±1
    rng.shuffle(lure_candidates)
    lure_positions = set(lure_candidates[:n_lures])
    
    items = []
    for i in range(length):
        if i < n_level:
            items.append(rng.choice(CONSONANTS))
        elif i in target_positions:
            ref = items[i - n_level]
            if transform:
                items.append(_NEXT_CONSONANT[ref])
            else:
                items.append(ref)
        elif i in lure_positions:
            # Lure: match at N-1 or N+1, but NOT at N
            ref_n = items[i - n_level]
            offsets = []
            if i - n_level - 1 >= 0:
                offsets.append(items[i - n_level - 1])  # N+1 back
            if i - n_level + 1 < i:
                offsets.append(items[i - n_level + 1])  # N-1 back
            # Pick a lure letter that differs from ref_n
            valid_lures = [l for l in offsets if l != ref_n]
            if transform:
                valid_lures = [l for l in valid_lures if l != _NEXT_CONSONANT[ref_n]]
            if valid_lures:
                items.append(rng.choice(valid_lures))
            else:
                # Fallback: random non-matching
                avoid = {ref_n}
                if transform:
                    avoid.add(_NEXT_CONSONANT[ref_n])
                pool = [c for c in CONSONANTS if c not in avoid]
                items.append(rng.choice(pool))
        else:
            # Non-target, non-lure
            ref_n = items[i - n_level]
            avoid = {ref_n}
            if transform:
                avoid.add(_NEXT_CONSONANT[ref_n])
            # Also avoid matching at N-1 and N+1 (don't accidentally create lures)
            if i - n_level - 1 >= 0:
                avoid.add(items[i - n_level - 1])
            if i - n_level + 1 < i:
                avoid.add(items[i - n_level + 1])
            pool = [c for c in CONSONANTS if c not in avoid]
            if not pool:
                pool = [c for c in CONSONANTS if c != ref_n]
            items.append(rng.choice(pool))
    
    # Build trials
    trials = []
    for i in range(length):
        if i < n_level:
            trial_type = "filler"
            is_target = False
        elif i in target_positions:
            trial_type = "target"
            is_target = True
        elif i in lure_positions:
            trial_type = "lure"
            is_target = False
        else:
            trial_type = "non_target"
            is_target = False
        
        trials.append({
            "position": i,
            "letter": items[i],
            "type": trial_type,
            "is_target": is_target,
            "n_back_letter": items[i - n_level] if i >= n_level else None,
            "correct_response": "YES" if is_target else "NO",
            "quartile": i * 4 // length,
        })
    
    stats = {
        "n_back": n_level,
        "length": length,
        "transform": transform,
        "targets": sum(1 for t in trials if t["type"] == "target"),
        "lures": sum(1 for t in trials if t["type"] == "lure"),
        "non_targets": sum(1 for t in trials if t["type"] == "non_target"),
    }
    
    return {"sequence": trials, "n_back": n_level, "transform": transform, "stats": stats}


# Generate all conditions
NBACK_V3 = {
    "2back": generate_nback_v3(2, 60, "nback_v3_2back"),
    "3back": generate_nback_v3(3, 60, "nback_v3_3back"),
    "4back": generate_nback_v3(4, 80, "nback_v3_4back"),
    "4back_transform": generate_nback_v3(4, 80, "nback_v3_4back_transform", transform=True),
    "5back": generate_nback_v3(5, 80, "nback_v3_5back"),
    "5back_transform": generate_nback_v3(5, 80, "nback_v3_5back_transform", transform=True),
}


# --- Inlined task ---
"""
N-back Working Memory v3 — Harder variant targeting LLM ceiling effects.

Changes from v2:
- Dropped trivial 1-back. Levels: 2, 3, 4, 5-back.
- Batch presentation: model sees sequence segments, must answer for ALL marked
  positions. The N-back reference letter is NOT revealed in the prompt.
- Transformation N-back: at 4-back and 5-back, half the conditions use a
  "next consonant" rule instead of identity matching.
- Lure trials (~12%): letters match at N±1 but NOT at N.
- Longer sequences: 80 items for 4-back and 5-back.
- Scoring: weighted d-prime, heavier weight on harder conditions.

Cognitive Science Basis:
- N-back (Kirchner, 1958; Owen et al., 2005)
- Dual/transformation N-back (Jaeggi et al., 2008)
- Lure trials increase false alarm rates (Kane et al., 2007)
"""

import kaggle_benchmarks as kbench
import re
import json as _json
import numpy as np
from dataclasses import dataclass
# (data inlined above)


def _safe_log(data): print(_json.dumps(data, indent=2, default=str))


def _strip_think(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def _parse_yes_no(raw: str, expected_count: int) -> list:
    """Parse YES/NO responses from model output."""
    raw = _strip_think(raw)
    raw = re.sub(r'//.*', '', raw)
    # Try JSON array
    try:
        m = re.search(r'\[.*\]', raw, re.DOTALL)
        if m:
            arr = _json.loads(m.group())
            if len(arr) == expected_count:
                return [str(x).strip().upper() for x in arr]
    except Exception:
        pass
    # Fallback: extract YES/NO tokens
    tokens = re.findall(r'\b(YES|NO|yes|no|Yes|No)\b', raw)
    result = [t.upper() for t in tokens]
    return result[:expected_count]


def run_nback_condition(llm, data: dict, condition_name: str) -> list:
    """Run one N-back condition with batch segment presentation."""
    seq = data["sequence"]
    n = data["n_back"]
    transform = data["transform"]
    segment_size = 10
    letters = [item["letter"] for item in seq]
    all_results = []

    for seg_start in range(0, len(seq), segment_size):
        seg_end = min(seg_start + segment_size, len(seq))
        eval_items = [item for item in seq[seg_start:seg_end] if item["position"] >= n]
        if not eval_items:
            continue

        # Build display — show FULL sequence up to this segment
        display_lines = []
        for i in range(seg_end):
            letter = letters[i]
            marker = " <-- respond" if seg_start <= i < seg_end and i >= n else ""
            display_lines.append(f"  [{i:2d}] {letter}{marker}")

        positions = [item["position"] for item in eval_items]

        if transform:
            rule_text = (
                f"Rule: For each marked position, answer YES if the letter is the "
                f"NEXT CONSONANT in the alphabet after the letter exactly {n} positions "
                f"earlier (B→C→D→F→G→H→J→K→L→M→N→P→Q→R→S→T→V→W→X→Z→B). "
                f"Answer NO otherwise."
            )
        else:
            rule_text = (
                f"Rule: For each marked position, answer YES if the letter is the "
                f"SAME as the letter exactly {n} positions earlier. Answer NO otherwise."
            )

        with kbench.chats.new(f"{condition_name}_seg{seg_start}"):
            prompt = (
                f"**{n}-Back {'Transform ' if transform else ''}Task — "
                f"Segment {seg_start // segment_size + 1}**\n\n"
                f"{rule_text}\n\n"
                f"Sequence so far:\n"
                + "\n".join(display_lines) + "\n\n"
                f"For positions {positions}, respond with ONLY a JSON array of "
                f"YES/NO strings. Example: [\"YES\", \"NO\", \"NO\", ...]\n"
                f"Give exactly {len(eval_items)} responses."
            )

            raw = llm.prompt(prompt)
            responses = _parse_yes_no(raw, len(eval_items))

            while len(responses) < len(eval_items):
                responses.append("NO")

            for item, resp in zip(eval_items, responses):
                correct = (resp == item["correct_response"])
                all_results.append({
                    "position": item["position"],
                    "letter": item["letter"],
                    "type": item["type"],
                    "correct_response": item["correct_response"],
                    "model_response": resp,
                    "correct": correct,
                    "is_hit": resp == "YES" and item["correct_response"] == "YES",
                    "is_miss": resp == "NO" and item["correct_response"] == "YES",
                    "is_false_alarm": resp == "YES" and item["correct_response"] == "NO",
                    "quartile": item["quartile"],
                })

    return all_results


def _dprime(hits, misses, fa, cr):
    """Compute d-prime with log-linear correction."""
    hr = (hits + 0.5) / (hits + misses + 1)
    far = (fa + 0.5) / (fa + cr + 1)
    # Rational approx of inverse normal CDF (Abramowitz & Stegun 26.2.23)
    def _ppf(p):
        import math
        if p <= 0.0001: return -3.7
        if p >= 0.9999: return 3.7
        if p == 0.5: return 0.0
        if p > 0.5:
            return -_ppf_low(1.0 - p)
        return _ppf_low(p)
    def _ppf_low(p):
        # For p < 0.5: returns negative z-score
        import math
        t = math.sqrt(-2.0 * math.log(p))
        # Rational approximation (Hastings, 1955)
        z = t - (2.515517 + 0.802853*t + 0.010328*t*t) / \
                (1.0 + 1.432788*t + 0.189269*t*t + 0.001308*t*t*t)
        return -z  # negative for p < 0.5
    return round(_ppf(hr) - _ppf(far), 4)


@kbench.task(name="N-Back Working Memory")
def exec_func_nback(llm) -> float:
    """
    N-back Working Memory v3.
    
    Tests working memory updating across N=2,3,4,5 with lure trials
    and transformation rules at higher N levels.
    
    Score = weighted normalized d-prime across conditions.
    """
    conditions = [
        ("2back", NBACK_V3["2back"], 0.08),
        ("3back", NBACK_V3["3back"], 0.15),
        ("4back", NBACK_V3["4back"], 0.15),
        ("4back_transform", NBACK_V3["4back_transform"], 0.20),
        ("5back", NBACK_V3["5back"], 0.15),
        ("5back_transform", NBACK_V3["5back_transform"], 0.27),
    ]
    
    condition_results = {}
    for cond_name, cond_data, weight in conditions:
        results = run_nback_condition(llm, cond_data, cond_name)
        
        hits = sum(1 for r in results if r["is_hit"])
        misses = sum(1 for r in results if r["is_miss"])
        fa = sum(1 for r in results if r["is_false_alarm"])
        cr = sum(1 for r in results if r["correct"] and r["correct_response"] == "NO")
        
        dp = _dprime(hits, misses, fa, cr)
        accuracy = sum(1 for r in results if r["correct"]) / max(len(results), 1)
        
        # Lure-specific false alarm rate
        lure_items = [r for r in results if r["type"] == "lure"]
        lure_fa = sum(1 for r in lure_items if r["model_response"] == "YES") / max(len(lure_items), 1)
        
        condition_results[cond_name] = {
            "d_prime": dp,
            "accuracy": round(accuracy, 4),
            "hits": hits, "misses": misses,
            "false_alarms": fa, "correct_rejections": cr,
            "lure_false_alarm_rate": round(lure_fa, 4),
            "n_items": len(results),
            "weight": weight,
        }
    
    # Composite: weighted normalized d-prime (d'=0→0, d'=4→1)
    score = sum(
        w * float(np.clip(condition_results[name]["d_prime"] / 4.0, 0, 1))
        for name, _, w in conditions
    )
    score = round(float(np.clip(score, 0, 1)), 4)
    
    # Logging
    print(f"\n{'='*60}")
    print(f"N-BACK WORKING MEMORY v3 RESULTS")
    print(f"{'='*60}")
    for name, _, w in conditions:
        cr = condition_results[name]
        print(f"\n  {name} (weight={w}):")
        print(f"    d'={cr['d_prime']:.3f}  acc={cr['accuracy']:.2%}  "
              f"hits={cr['hits']} miss={cr['misses']} FA={cr['false_alarms']} CR={cr['correct_rejections']}")
        print(f"    lure FA rate={cr['lure_false_alarm_rate']:.2%}")
    print(f"\n  COMPOSITE SCORE: {score:.4f}")
    print(f"{'='*60}")
    
    _safe_log({
        "benchmark": "N-back v3",
        "conditions": condition_results,
        "composite_score": score,
    })
    
    return score


In [ ]:
exec_func_nback.run(llm=kbench.llm)
